# Como foi feito o projeto da habitação em Portugal, passo a passo

**A pergunta.** Quanto é que os preços da habitação em Portugal se afastaram dos rendimentos, em comparação com seis países
e com a UE, e quem sente esse afastamento?

**Como está organizado este notebook.** Segue o projeto pela ordem em que foi construído:

| Passo | O quê | Ferramenta |
|---|---|---|
| 1 | Descarregar cinco tabelas do Eurostat pela API de estatísticas | Biblioteca padrão do Python |
| 2 | Carregar os ficheiros em bruto e verificar a cobertura e as marcas de estado | pandas |
| 3 | Modelar os índices em dbt: trimestres, preços reais, preço/rendimento | dbt sobre DuckDB |
| 4 | Responder à pergunta com gráficos interativos | pandas + Altair |
| 5 | Conferir o pandas com os marts do dbt e com os números de referência publicados | pandas + DuckDB |
| 6 | Publicar: painel web, Power BI e Qlik | Exportação JSON, PBIP, script Qlik |

Ao contrário dos outros projetos do portfólio, aqui os dados são **reais**: estatísticas oficiais do Eurostat para Portugal,
Espanha, Irlanda, Itália, França, Alemanha, Países Baixos e UE-27, descarregadas a 24 de outubro de 2026.

**Os gráficos são interativos**: alterne entre preços reais e nominais e clique num país para o destacar ou para comandar
o gráfico ligado. São feitos com [Altair](https://altair-viz.github.io/); abra a
[versão interativa](https://guilhermerigonato.com/pt-housing-analytics/notebook/pt/) para os usar (o GitHub mostra
os notebooks sem JavaScript). O código e os nomes das colunas ficam em inglês, como no repositório.

**Para correr:** `pip install pandas altair duckdb dbt-duckdb`; depois, na raiz do repositório,
`python scripts/get_data.py` e `dbt build --profiles-dir .`, e execute todas as células (alguns segundos).

In [1]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
import altair as alt

RAW = Path("../data/raw")
pd.set_option("display.float_format", "{:,.1f}".format)
NAMES = {"PT": "Portugal", "ES": "Espanha", "IE": "Irlanda", "IT": "Itália", "FR": "França", "DE": "Alemanha",
         "NL": "Países Baixos", "EU27_2020": "UE-27"}
ORDER = list(NAMES.values())
BLUE, ORANGE, GREEN, GREY, MUTED, GRID = "#2a78d6", "#eb6834", "#1baf7a", "#c9c8c3", "#52514e", "#e4e3df"
PALETTE = alt.Scale(domain=ORDER, range=[ORANGE, BLUE, GREEN, "#9a6fd0", "#c2a34a", "#5aa5c9", "#d77fa1", MUTED])

@alt.theme.register("portfolio", enable=True)
def portfolio_theme():
    axis = {"gridColor": GRID, "domainColor": MUTED, "tickColor": MUTED, "labelColor": MUTED,
            "titleColor": MUTED, "labelFontSize": 11, "titleFontSize": 11, "titleFontWeight": "normal"}
    return {"config": {"view": {"stroke": None}, "axis": axis, "legend": {"labelColor": MUTED, "titleColor": MUTED},
                       "title": {"anchor": "start", "fontSize": 13, "color": "#0b0b0b"},
                       "font": "system-ui, -apple-system, Segoe UI, sans-serif"}}

## Passo 1. Descarregar os dados

[`scripts/get_data.py`](../scripts/get_data.py) chama a API de difusão do Eurostat, que responde em JSON-stat:
uma lista plana de valores e, para cada dimensão, a lista de códigos e as respetivas posições. O script desdobra isso num
CSV longo (uma coluna por dimensão, depois período, valor e marca de estado) usando só a biblioteca padrão.

| Tabela | O quê | Frequência |
|---|---|---|
| `prc_hpi_q` | Índice de preços da habitação, 2015 = 100, todos os alojamentos, novos e existentes | Trimestral |
| `prc_hicp_minr` | Preços no consumidor (IHPC), total e rendas efetivas (ECOICOP ver. 2) | Mensal |
| `ilc_lvho07a` | Taxa de sobrecarga com despesas de habitação, total e abaixo de 60% do rendimento mediano | Anual (inquérito) |
| `ilc_di03` | Rendimento líquido equivalente mediano | Anual (inquérito) |
| `sts_cobp_a` | Licenças de construção, fogos em edifícios residenciais | Anual |

Um pormenor que conta: em 2026 o Eurostat congelou a tabela antiga do IHPC (`prc_hicp_midx`) em dezembro de 2025 e passou a
`prc_hicp_minr`. Um projeto que mantivesse o código antigo parava em 2025 sem dar erro.

In [2]:
raw = {f.stem: pd.read_csv(f, dtype=str) for f in sorted(RAW.glob("*.csv"))}
pd.DataFrame({"rows": {k: len(v) for k, v in raw.items()},
              "columns": {k: ", ".join(v.columns) for k, v in raw.items()}})

,rows,columns
ilc_di03,446,"freq, age, sex, statinfo, unit, geo, time_peri..."
ilc_lvho07a,489,"freq, unit, rskpovth, age, sex, geo, time_peri..."
prc_hicp_minr,5810,"freq, unit, coicop18, geo, time_period, obs_va..."
prc_hpi_q,1881,"freq, purchase, unit, geo, time_period, obs_va..."
sts_cobp_a,4140,"freq, indic_bt, cpa2_1, s_adj, unit, geo, time..."


## Passo 2. Carregar os ficheiros em bruto e verificá-los

Os dados oficiais têm as suas armadilhas: valores provisórios (`p`) e estimados (`e`), séries que começam em anos
diferentes e falhas. As verificações abaixo dizem em que é que o modelo se pode apoiar.

In [3]:
hpi = raw["prc_hpi_q"].query("unit == 'I15_Q'").copy()
hpi["quarter_start"] = pd.PeriodIndex(hpi["time_period"].str.replace("-", ""), freq="Q").start_time
hpi["value"] = hpi["obs_value"].astype(float)
hpi = hpi[hpi["quarter_start"] >= "2005-01-01"]

coverage = hpi[hpi["purchase"] == "TOTAL"].groupby("geo").agg(
    first=("quarter_start", "min"), last=("quarter_start", "max"), quarters=("value", "count"),
    provisional=("obs_flag", lambda f: (f == "p").sum()))
coverage.index = coverage.index.map(NAMES)
coverage

,first,last,quarters,provisional
geo,,,,
Alemanha,2005-01-01,2026-04-01,86,2
Espanha,2005-10-01,2026-04-01,83,0
UE-27,2005-01-01,2026-04-01,86,0
França,2005-01-01,2026-04-01,86,1
Irlanda,2005-01-01,2026-04-01,86,1
Itália,2010-01-01,2026-04-01,66,1
Países Baixos,2005-01-01,2026-04-01,86,1
Portugal,2008-01-01,2026-04-01,74,0


In [4]:
flags = pd.concat({k: v["obs_flag"].fillna("(none)").value_counts() for k, v in raw.items()}, axis=1).fillna(0).astype(int)
flags.index.name = "status flag"
flags   # p = provisório, e = estimado, b = quebra de série, u = baixa fiabilidade

,ilc_di03,ilc_lvho07a,prc_hicp_minr,prc_hpi_q,sts_cobp_a
status flag,,,,,
(none),404,450,5330,1864,3652
b,30,24,0,0,0
e,12,15,480,0,180
p,0,0,0,17,136
i,0,0,0,0,172


## Passo 3. Modelar os índices em dbt

O SQL está em [`models/`](../models). As views de staging tipam e filtram as tabelas em bruto; os marts formam um pequeno
esquema em estrela (`dim_country`, `dim_quarter`, `fct_house_prices`, `fct_consumer_prices`, `fct_living_conditions`,
`fct_building_permits`) e dois marts respondem à pergunta:

* `mart_house_prices_real`: o IHPC mensal passa a média trimestral (só trimestres completos) e depois
  **preço real da habitação = índice de preços da habitação ÷ IHPC total × 100**, mais rendas e variações homólogas.
* `mart_affordability`: **preço/rendimento = média anual do índice de preços da habitação ÷ rendimento mediano indexado a 2015 × 100**.
  O EU-SILC pergunta pelo rendimento do ano anterior, por isso o inquérito do ano *t* é alinhado com o rendimento do ano *t − 1*
  antes de ser comparado com os preços. É por isso que o último ano de preço/rendimento é 2024 e não 2025.

O `dbt build` corre 38 nós (modelos e testes: chaves, relações, valores aceites e uma verificação de que cada índice
tem média 100 no ano base) sem falhas. Os mesmos passos em pandas:

In [5]:
cpi = raw["prc_hicp_minr"].query("unit == 'I15'").copy()
cpi["month"] = pd.to_datetime(cpi["time_period"].str.replace("M", "-") + "-01")
cpi["value"] = cpi["obs_value"].astype(float)
cpi = cpi[cpi["month"] >= "2005-01-01"]
cpi["quarter_start"] = cpi["month"].dt.to_period("Q").dt.start_time
cpi_q = cpi.groupby(["geo", "coicop18", "quarter_start"])["value"].agg(["mean", "count"])
cpi_q = cpi_q[cpi_q["count"] == 3]["mean"].unstack("coicop18").rename(columns={"TOTAL": "cpi", "CP041": "rent"})

q = hpi[hpi["purchase"] == "TOTAL"].set_index(["geo", "quarter_start"])["value"].rename("nominal").to_frame().join(cpi_q, how="inner")
q["real"] = q["nominal"] / q["cpi"] * 100
q = q.reset_index()
q["country"] = q["geo"].map(NAMES)
q["nominal_yoy"] = q.groupby("geo")["nominal"].pct_change(4)
q["real_yoy"] = q.groupby("geo")["real"].pct_change(4)
last = q["quarter_start"].max()
q[q["quarter_start"] == last].set_index("country")[["nominal", "real", "nominal_yoy", "real_yoy"]].sort_values("nominal", ascending=False) \
    .style.format({"nominal": "{:.1f}", "real": "{:.1f}", "nominal_yoy": "{:+.1%}", "real_yoy": "{:+.1%}"})

,nominal,real,nominal_yoy,real_yoy
country,,,,
Portugal,301.4,232.4,+16.5%,+12.9%
Países Baixos,222.6,159.2,+4.3%,+1.5%
Irlanda,205.9,163.6,+6.0%,+2.4%
Espanha,200.6,152.8,+12.1%,+8.3%
UE-27,168.6,123.0,+4.7%,+1.5%
Alemanha,153.5,113.5,+0.6%,-2.0%
França,125.5,98.4,-0.8%,-3.1%
Itália,121.0,94.0,+4.0%,+0.9%


## Passo 4. Responder à pergunta

### Preços da habitação desde 2015, face aos outros países

**Alterne entre real e nominal** com os botões por baixo do gráfico; **clique num país na legenda** para o
destacar. Os preços reais retiram a inflação no consumidor, e isso conta: o índice de Portugal triplicou em termos nominais e
mais do que duplicou em termos reais.

In [6]:
long = q[q["quarter_start"] >= "2015-01-01"].melt(id_vars=["country", "quarter_start"], value_vars=["nominal", "real"],
                                                  var_name="measure", value_name="index")
measure = alt.param(name="measure", value="real", bind=alt.binding_radio(options=["real", "nominal"], name="Preços: "))
country = alt.selection_point(fields=["country"], bind="legend", value=[{"country": "Portugal"}])
alt.Chart(long).transform_filter("datum.measure == measure").mark_line(strokeWidth=2).encode(
    x=alt.X("quarter_start:T", title=None),
    y=alt.Y("index:Q", title="Índice de preços da habitação, 2015 = 100"),
    color=alt.Color("country:N", scale=PALETTE, title="País (clique)"),
    opacity=alt.condition(country, alt.value(1), alt.value(0.15)),
    strokeWidth=alt.condition(country, alt.value(3), alt.value(1.5)),
    tooltip=["country", alt.Tooltip("quarter_start:T", title="Trimestre", format="%Y T%q"), alt.Tooltip("index:Q", format=".1f")],
).add_params(measure, country).properties(width=560, height=280, title="Preços da habitação, 2015 = 100")

alt.Chart(...)

### Preços, rendas e custo de vida em Portugal

As rendas do IHPC abrangem todos os arrendamentos, incluindo contratos antigos, por isso sobem muito mais devagar do que as
rendas pedidas em contratos novos; ainda assim, a distância face aos preços das casas é grande. Passe o cursor para ver os valores.

In [7]:
pt = q[(q["geo"] == "PT") & (q["quarter_start"] >= "2015-01-01")].melt(
    id_vars="quarter_start", value_vars=["nominal", "rent", "cpi"], var_name="series", value_name="index")
pt["series"] = pt["series"].map({"nominal": "Preços da habitação", "rent": "Rendas (IHPC)", "cpi": "Preços no consumidor (total)"})
hover = alt.selection_point(fields=["quarter_start"], nearest=True, on="pointerover", empty=False)
base = alt.Chart(pt).encode(x=alt.X("quarter_start:T", title=None))
lines = base.mark_line(strokeWidth=2).encode(
    y=alt.Y("index:Q", title="Índice, 2015 = 100"),
    color=alt.Color("series:N", scale=alt.Scale(range=[ORANGE, BLUE, MUTED]), title=None))
points = base.mark_point(filled=True, size=60).encode(
    y="index:Q", color=alt.Color("series:N", legend=None),
    opacity=alt.condition(hover, alt.value(1), alt.value(0)),
    tooltip=[alt.Tooltip("quarter_start:T", format="%Y T%q", title="Trimestre"), "series", alt.Tooltip("index:Q", format=".1f")],
).add_params(hover)
rule = base.mark_rule(color=GREY).encode(opacity=alt.condition(hover, alt.value(1), alt.value(0))).transform_filter(hover)
(lines + points + rule).properties(width=560, height=260, title="Portugal: preços da habitação, rendas e preços no consumidor")

alt.LayerChart(...)

### Os preços afastaram-se dos rendimentos?

**Clique num país** à esquerda: o gráfico da direita mostra o seu índice de preços da habitação face ao rendimento mediano,
ambos indexados a 2015. Onde a linha laranja se afasta da azul, a habitação ficou menos acessível.

In [8]:
ann = q.assign(year=q["quarter_start"].dt.year).groupby(["geo", "year"])["nominal"].agg(["mean", "count"])
ann = ann[ann["count"] == 4]["mean"].rename("hpi").reset_index()
inc = raw["ilc_di03"].query("statinfo == 'MED_EI' and unit == 'EUR' and age == 'TOTAL' and sex == 'T'").copy()
inc["year"] = inc["time_period"].astype(int) - 1                # o inquérito do ano t reporta o rendimento do ano t - 1
inc["income"] = inc["obs_value"].astype(float)
aff = ann.merge(inc[["geo", "year", "income"]], on=["geo", "year"])
aff["income_2015"] = aff["income"] / aff.groupby("geo")["income"].transform(lambda s: s[aff.loc[s.index, "year"] == 2015].iloc[0]) * 100
aff["pti"] = aff["hpi"] / aff["income_2015"] * 100
aff["country"] = aff["geo"].map(NAMES)
pti_last = aff[aff["year"] == aff["year"].max()].sort_values("pti", ascending=False)

pick = alt.selection_point(name="pick", fields=["country"], on="click", clear="dblclick", value=[{"country": "Portugal"}])
bars = alt.Chart(pti_last).mark_bar(cornerRadius=3).encode(
    y=alt.Y("country:N", sort="-x", title=None),
    x=alt.X("pti:Q", title=f"Índice preço/rendimento, {int(pti_last['year'].iloc[0])} (2015 = 100)"),
    color=alt.condition(pick, alt.value(ORANGE), alt.value(GREY)),
    tooltip=["country", alt.Tooltip("pti:Q", format=".1f", title="Preço/rendimento")],
).add_params(pick).properties(width=250, height=220, title="Clique num país")
rule100 = alt.Chart(pd.DataFrame({"x": [100]})).mark_rule(strokeDash=[3, 3], color=MUTED).encode(x="x:Q")

two = aff[aff["year"] >= 2015].melt(id_vars=["country", "year"], value_vars=["hpi", "income_2015"], var_name="series", value_name="index")
two["series"] = two["series"].map({"hpi": "Preços da habitação", "income_2015": "Rendimento mediano"})
chosen = "(pick.country && length(pick.country)) ? pick.country[0] : 'Portugal'"
lines = alt.Chart(two).transform_filter(f"datum.country == ({chosen})").mark_line(point=True, strokeWidth=2).encode(
    x=alt.X("year:O", title=None, axis=alt.Axis(labelAngle=0)),
    y=alt.Y("index:Q", title="Índice, 2015 = 100", scale=alt.Scale(zero=False)),
    color=alt.Color("series:N", scale=alt.Scale(range=[ORANGE, BLUE]), title=None),
    tooltip=["country", "series", "year", alt.Tooltip("index:Q", format=".1f")],
).properties(width=380, height=220, title=alt.Title(alt.expr(f"'Preços da habitação e rendimento mediano · ' + ({chosen})")))
alt.hconcat(bars + rule100, lines)

alt.HConcatChart(...)

### A construção acompanhou, e quem está sobrecarregado?

As licenças para casas novas em Portugal recuperaram do mínimo de 2014, mas continuam muito abaixo de 2005. A taxa de
sobrecarga é a parte das pessoas que gastam mais de 40% do rendimento disponível com a habitação. **Arraste sobre o gráfico das licenças**
para escolher anos; o gráfico da direita mostra as taxas de sobrecarga nesses anos.

In [9]:
per = raw["sts_cobp_a"].query("indic_bt == 'BPRM_DW' and cpa2_1 == 'CPA_F41001_X_410014' and unit == 'THS' and geo == 'PT'").copy()
per["year"] = per["time_period"].astype(int)
per["dwellings"] = per["obs_value"].astype(float) * 1000
ob = raw["ilc_lvho07a"].query("unit == 'PC' and age == 'TOTAL' and sex == 'T' and geo in ['PT', 'EU27_2020'] and rskpovth in ['TOTAL', 'B_60']").copy()
ob["year"] = ob["time_period"].astype(int)
ob["rate"] = ob["obs_value"].astype(float) / 100
ob["group"] = ob["geo"].map(NAMES) + " · " + ob["rskpovth"].map({"TOTAL": "todos", "B_60": "baixo rendimento"})

years = alt.selection_interval(encodings=["x"], value={"x": [2015, 2025]})
permits = alt.Chart(per).mark_bar(color=BLUE, size=11).encode(
    x=alt.X("year:Q", title=None, axis=alt.Axis(format="d"), scale=alt.Scale(domain=[2004.5, 2025.5])),
    y=alt.Y("dwellings:Q", title="Fogos licenciados", axis=alt.Axis(format="~s")),
    color=alt.condition(years, alt.value(BLUE), alt.value(GREY)),
    tooltip=["year", alt.Tooltip("dwellings:Q", format=",.0f")],
).add_params(years).properties(width=330, height=220, title="Portugal: fogos licenciados (arraste para escolher anos)")
burden = alt.Chart(ob).transform_filter(years).mark_line(point=True).encode(
    x=alt.X("year:Q", title="Ano do inquérito", axis=alt.Axis(format="d"), scale=alt.Scale(zero=False)),
    y=alt.Y("rate:Q", title="Sobrecarregados com a habitação", axis=alt.Axis(format="%")),
    color=alt.Color("group:N", title=None, legend=alt.Legend(orient="bottom", columns=2),
                    scale=alt.Scale(domain=["Portugal · todos", "Portugal · baixo rendimento", "UE-27 · todos", "UE-27 · baixo rendimento"],
                                    range=[ORANGE, "#f2b08f", MUTED, GREY])),
    strokeDash=alt.StrokeDash("group:N", legend=None),
    tooltip=["group", "year", alt.Tooltip("rate:Q", format=".1%")],
).properties(width=330, height=220, title="Taxa de sobrecarga nos anos escolhidos")
alt.hconcat(permits, burden)

alt.HConcatChart(...)

**Resposta.** Os preços da habitação em Portugal triplicaram desde 2015 (2,3 vezes descontada a inflação), a maior subida dos
oito; de 2015 a 2024 subiram 124%, enquanto o rendimento mediano subiu 65%, por isso o índice preço/rendimento (136) é o mais
alto do grupo. A construção recuperou, mas continua 45% abaixo de 2005. Não é na família média que a pressão se vê: só 6,3%
das pessoas estão sobrecarregadas, abaixo dos 7,7% da UE, porque a maioria tem casa própria sem crédito; entre as famílias de
baixo rendimento é cerca de uma em cada quatro.

## Passo 5. Conferir o pandas com os marts do dbt e os números de referência

O README publica números de referência para que as versões Power BI e Qlik possam ser verificadas depois da primeira carga.
Aqui o pandas, os marts do dbt e esses números têm de coincidir.

In [10]:
db = duckdb.connect("../data/housing.duckdb", read_only=True)
m_real = db.sql("select geo, quarter_start, nominal_hpi, real_hpi, nominal_yoy from mart_house_prices_real").df()
m_aff = db.sql("select geo, year, price_to_income_index from mart_affordability where price_to_income_index is not null").df()
db.close()

ptq = q[(q["geo"] == "PT") & (q["quarter_start"] == "2026-04-01")].iloc[0]
mq = m_real[(m_real["geo"] == "PT") & (m_real["quarter_start"] == pd.Timestamp("2026-04-01"))].iloc[0]
pti_pt = aff[(aff["geo"] == "PT") & (aff["year"] == 2024)]["pti"].iloc[0]
m_pti = m_aff[(m_aff["geo"] == "PT") & (m_aff["year"] == 2024)]["price_to_income_index"].iloc[0]
merged = q.merge(m_real, on=["geo", "quarter_start"])
recon = pd.DataFrame([
    ["PT índice de preços da habitação, 2026 T2 (nominal)", ptq["nominal"], mq["nominal_hpi"], 301.4],
    ["PT índice de preços da habitação, 2026 T2 (real)", ptq["real"], mq["real_hpi"], 232.4],
    ["PT variação homóloga, 2026 T2 (%)", ptq["nominal_yoy"] * 100, mq["nominal_yoy"] * 100, 16.5],
    ["PT índice preço/rendimento, 2024", pti_pt, m_pti, 136.3],
    ["PT fogos licenciados, 2025", per.loc[per["year"] == 2025, "dwellings"].iloc[0], None, 40_900],
    ["Maior diferença no índice real, qualquer país e trimestre", (merged["real"] - merged["real_hpi"]).abs().max(), 0, None],
], columns=["measure", "pandas", "dbt", "README"])
recon["match"] = [np.isclose(r.pandas, r.dbt if pd.notna(r.dbt) else r.pandas, atol=1e-6)
                  and (pd.isna(r.README) or abs(r.pandas - r.README) < 0.05 + 1e-9 * abs(r.README)) for r in recon.itertuples()]
assert recon["match"].all(), recon
recon

,measure,pandas,dbt,README,match
0,"PT índice de preços da habitação, 2026 T2 (nominal)",301.4,301.4,301.4,True
1,"PT índice de preços da habitação, 2026 T2 (real)",232.4,232.4,232.4,True
2,"PT variação homóloga, 2026 T2 (%)",16.5,16.5,16.5,True
3,"PT índice preço/rendimento, 2024",136.3,136.3,136.3,True
4,"PT fogos licenciados, 2025","40,900.0",NaN,"40,900.0",True
5,"Maior diferença no índice real, qualquer país e trimestre",0.0,0.0,NaN,True


O pandas e o dbt coincidem em todos os trimestres e países, e ambos batem com os números de referência publicados.

## Passo 6. Publicar

* **Painel web** ([`web/`](../web), online em [guilhermerigonato.com/pt-housing-analytics](https://guilhermerigonato.com/pt-housing-analytics/?lang=pt)):
  [`scripts/export.py`](../scripts/export.py) escreve JSON a partir dos marts; clicar num gráfico fixa um trimestre ou um ano
  em todos os painéis, e uma barra de país muda a comparação.
* **Power BI** ([`powerbi/`](../powerbi)) e **Qlik Sense** ([`qlik/`](../qlik)) carregam os ficheiros Parquet de
  [`exports/`](../exports), por isso nenhum dos dois precisa do DuckDB.

## Que biblioteca usar para gráficos interativos

| Biblioteca | Gráficos ligados | Resultado | Melhor para |
|---|---|---|---|
| **Altair** (Vega-Lite), usada aqui | Nativos: seleções e controlos partilhados entre gráficos | HTML estático, sem servidor | Relatórios que têm de abrir em qualquer lado |
| **Plotly** | Por gráfico; ligar gráficos exige Dash | HTML; uma app Dash para ligar | Gráficos individuais ricos, mapas |
| **Bokeh** | Seleção ligada através de uma fonte de dados partilhada | HTML ou um servidor Bokeh | Dispersões grandes, ferramentas à medida |
| **HoloViews + Panel** | `link_selections` numa só chamada | Notebook ou app Panel | Transformar um notebook numa pequena app |
| **Dash / Streamlit** | Qualquer, com callbacks em Python | Precisa de servidor | Ferramentas internas com dados ao vivo |